In [8]:
from pathlib import Path

from satchange.automated_hpo import run_automated_parameter_search
from satchange.run import resolve_parameters
from satchange.utils import load_config

cwd = Path.cwd().resolve()
project_root = cwd.parents[1] if cwd.parent.name == "notebooks" else cwd
config = load_config(project_root / "configs/siam_mit_b3_diff_resized.yaml")
dataset_root = project_root / config["dataset"]["prepared_directory"]
HPO_ROOT = project_root / "artifacts" / "hyperparameter_search" / "heavy" / "heavy_resized"

GPU_IDS = [0, 1, 2, 3]
THRESHOLDS = [round(value / 100, 2) for value in range(5, 96, 5)]


In [9]:
HPO_STAGES = [
    ("01_batch_lr", {
        "batch_size": [4, 8, 12, 16],
        "learning_rate": [3e-5, 1e-4, 3e-4],
    }),

    ("02_lr", {
        "learning_rate": [1e-4, 3e-4, 6e-4],
    }),

    ("03_loss", {
        "positive_weight": [5, 10],
        "loss_weights": [(0.3, 0.7), (0.5, 0.5)],
    }),

    ("04_wd_scheduler", {
        "weight_decay": [1e-3, 1e-2],
        "scheduler": ["none", "reduce_on_plateau"],
    }),
]

In [10]:
from pathlib import Path

BEST_CHECKPOINTS = [
    Path("../../artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_000/best.pt"),
    Path("../../artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_001/best.pt"),
    Path("../../artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_002/best.pt"),
    Path("../../artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_003/best.pt")
]

In [11]:
summary = run_automated_parameter_search(
    stages=HPO_STAGES,
    base_parameters=resolve_parameters(config),
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,

    number_of_epochs=60,
    patience=20,
    min_epochs=40,
    min_delta=0.001,

    output_root=HPO_ROOT,
    search_name="heavy_resized_from_best_60ep_v3",
    gpu_ids=GPU_IDS,

    checkpoint_interval=10,

    resume_incomplete=True,
    resume_from=BEST_CHECKPOINTS,

    f1_gap=0.04,
    iou_gap=0.05,
    seeds=None,
)

best = summary["selected_by_mean_validation_f1"]
print("Parameters:", best["parameters"])
print("Validation F1:", best["mean_f1"], "+/-", best["std_f1"])
print("Validation IoU:", best["mean_iou"], "+/-", best["std_iou"])
print("Results:", summary["search_directory"])


src/satchange/optimization.py:255: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_000/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(
src/satchange/optimization.py:255: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_001/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(
src/satchange/optimization.py:255: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_002/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(


01_batch_lr: new=12, reused=0
01_batch_lr: resume trial 000 after epoch 51
01_batch_lr: resume trial 001 after epoch 59
01_batch_lr: resume trial 002 after epoch 60
01_batch_lr: resume trial 003 after epoch 57


src/satchange/optimization.py:255: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_003/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(


Trials: 4
GPUs: [0, 1, 2, 3]
Single-GPU memory limit: 80000 MB

[START] Trial 000 -> GPU 3 | attempt=1
seed=42 | lr=3e-05 | wd=0.01 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=256 | attention_dropout=0.0 | decoder_channels=128 | drop_path_rate=0.1 | dropout=0.0 | encoder_depths=[3, 4, 18, 3] | in_channels=3 | norm_groups=8 | out_channels=1 | pretrained_backbone=False | refinement_channels=[64, 32] | use_checkpointing=False | scheduler=none
epochs=60

[START] Trial 001 -> GPU 0 | attempt=1
seed=42 | lr=3e-05 | wd=0.01 | pw=10 | BCE/Dice=0.3/0.7 | batch=8 | image_size=256 | attention_dropout=0.0 | decoder_channels=128 | drop_path_rate=0.1 | dropout=0.0 | encoder_depths=[3, 4, 18, 3] | in_channels=3 | norm_groups=8 | out_channels=1 | pretrained_backbone=False | refinement_channels=[64, 32] | use_checkpointing=False | scheduler=none
epochs=60

[START] Trial 002 -> GPU 2 | attempt=1
seed=42 | lr=3e-05 | wd=0.01 | pw=10 | BCE/Dice=0.3/0.7 | batch=12 | image_size=256 | attention_dropout

src/satchange/optimization.py:255: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_003/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(
src/satchange/optimization.py:255: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_000/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(
src/satchange/optimization.py:255: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_002/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(
src/satchange/optimization.py:255: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_e


Trial 2 finished on GPU 2
Parameters: seed=42 | lr=3e-05 | wd=0.01 | pw=10 | BCE/Dice=0.3/0.7 | batch=12 | image_size=256 | attention_dropout=0.0 | decoder_channels=128 | drop_path_rate=0.1 | dropout=0.0 | encoder_depths=[3, 4, 18, 3] | in_channels=3 | norm_groups=8 | out_channels=1 | pretrained_backbone=False | refinement_channels=[64, 32] | use_checkpointing=False | scheduler=none
Model parameters: total=45,772,961 | trainable=45,772,961
F1: 0.514096
IoU: 0.345982
Epochs: 60/60 (max_epochs)
GPU hours: 2.192
Resumed from checkpoint: artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_from_best_60ep_v3/01_batch_lr_9986b337b852/siam_mit_b3_diff/trial_003/last.pt
Saved epoch: 57
Next epoch: 58
Resumed from checkpoint: artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_from_best_60ep_v3/01_batch_lr_9986b337b852/siam_mit_b3_diff/trial_000/last.pt
Saved epoch: 51
Next epoch: 52
Resumed from checkpoint: artifacts/hyperparameter_search/heavy/heavy_resized/heavy_r